In [1]:
# goal: build a 3D encoder for each shell's coordinate probability dist.
# treat each possible x, y, z coordinate using wavenet style
# generate a "radialshell" class (naming can come later)
# holds radius attribute (its radius) and probability distributions
# i'm cooking

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from map4 import MAP4


In [22]:
# load out sample set
import ast
import re
def parse_3d_col(s):
    arrays = re.findall(r"array\(\[(.*?)\]\)", s)
    return [np.array([float(x) for x in a.split(',')]) for a in arrays]
AA3D_df = pd.read_csv("notebook_database/AA3D_df.csv")
AA3D_df["AA_IDs"] = AA3D_df['AA_IDs'].apply(ast.literal_eval)
AA3D_df["3d_struct"] = AA3D_df["3d_struct"].apply(parse_3d_col)

samples = AA3D_df.iloc[:100]
test_sample = AA3D_df.iloc[42]

In [2]:
AA_IDS = [
    'A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y','VOID',
]

# molecular weights are in kilodaltons
MOLECULAR_WEIGHTS = {
    'A': 0.09,
    'C': 0.12,
    'D': 0.13,
    'E': 0.15,
    'F': 0.17,
    'G': 0.08,
    'H': 0.16,
    'I': 0.13,
    'K': 0.15,
    'L': 0.13,
    'M': 0.15,
    'N': 0.13,
    'P': 0.12,
    'Q': 0.15,
    'R': 0.17,
    'S': 0.11,
    'T': 0.12,
    'V': 0.12,
    'W': 0.20,
    'Y': 0.18,
    'VOID': 0,
}

# net charges at physiological pH ~7.4
NET_CHARGES = {
    'A': 0,
    'C': 0,
    'D': -1,
    'E': -1,
    'F': 0,
    'G': 0,
    'H': 0,
    'I': 0,
    'K': +1,
    'L': 0,
    'M': 0,
    'N': 0,
    'P': 0,
    'Q': 0,
    'R': +1,
    'S': 0,
    'T': 0,
    'V': 0,
    'W': 0,
    'Y': 0,
    'VOID': 0,
}

# isoelectric point pHs from peptide web
# NOTE: will retrain model since I was using old textbook values with one outdated value -> shouldn't affect model accuracy much
ISOELECTRIC_PTS = {
    'A': 6.02,
    'C': 5.02,
    'D': 2.98,
    'E': 3.22,
    'F': 5.48,
    'G': 5.97,
    'H': 7.59,
    'I': 5.98,
    'K': 9.47,
    'L': 5.98,
    'M': 5.75,
    'N': 5.41,
    'P': 6.30,
    'Q': 5.65,
    'R': 10.76,
    'S': 5.68,
    'T': 5.60,
    'V': 5.97,
    'W': 5.94,
    'Y': 5.66,
    'VOID': 0,
}

# hydrophobicity levels - pH 7
# +ve values and up = hydrophobic
# around 0 = neutral
# -ve values = hydrophilic
HYDROPHOBICITY_IDXS = {
    'A': 41,
    'C': 49,
    'D': -55,
    'E': -31,
    'F': 100,
    'G': 0,
    'H': 8,
    'I': 100,
    'K': -23,
    'L': 97,
    'M': 74,
    'N': -28,
    'P': -46,
    'Q': -10,
    'R': -14,
    'S': -5,
    'T': 13,
    'V': 76,
    'W': 97,
    'Y': 63,
    'VOID': 0,
}


# individual half lives
HALF_LIFE = {
    'A': 4.4,
    'C': 1.2,
    'D': 1.1,
    'E': 1,
    'F': 1,
    'G': 30,
    'H': 3.5,
    'I': 20,
    'K': 1.3,
    'L': 5.5,
    'M': 30,
    'N': 1.4,
    'P': 21,
    'Q': 0.8,
    'R': 1,
    'S': 1.9,
    'T': 7.2,
    'V': 100,
    'W': 2.8,
    'Y': 2.8,
    'VOID': 0,
}


# pseudo molecular fingerprint accounting for biochemical / functional group standout features
# check function_fp_scheme.txt for more understanding
FUNCTIONAL_FP = {
    "A": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "G": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "D": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "E": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "K": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "R": [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "H": [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "N": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "Q": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "C": [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    "M": [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
    "F": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    "Y": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
    "W": [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    "L": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "I": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "V": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "S": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "T": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "P": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    "VOID": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

In [3]:
# feature engineering modules

In [4]:
class NAAnoEng:
    """Run this everytime we need a new set of feature vectors"""
    def __init__(self, verbose=False):
        self.nAAno_emb = {}   # basically just make it easier to embed down the line
        self.verbose = verbose

    def initialize(self):
        self.nAAno_vectors = {aa_id: get_embedding(aa_id) for aa_id in AA_IDS}

        if self.verbose:
            print("NAAnoEng initialized")
        return True

    def get_aa_id(self, naano_vector):
        aa_id = None
        for code, key in self.nAAno_vectors.items():
            if key == naano_vector:
                aa_id = code
                break

        if aa_id is None:
            raise ValueError(f"Feature vector {naano_vector} presented does not exist")

        return aa_id

    def are_you_really_an_engineer(self):
        """
        Questions if they are a certified Nanotech Engineer
        :return:
        """
        return False


def get_embedding(aa_id: str):
    """
    use this in these two scenarios:
    \n    - generating embeddings after updating feature vector
    \n    - inference
    :param aa_id: single letter amino acid reference code
    :returns: feature vector representing a given (valid) amino acid
    """
    # sanity check
    if aa_id not in AA_IDS:
        raise ValueError(f"{aa_id} not in valid AA ID list")

    # embedding scheme, MAKE SURE TO UPDATE THIS IF YOU EVER UPDATE NAANOLIBRARY
    naano_vector = [
        MOLECULAR_WEIGHTS[aa_id],
        NET_CHARGES[aa_id],
        ISOELECTRIC_PTS[aa_id],
        HYDROPHOBICITY_IDXS[aa_id],
        HALF_LIFE[aa_id],
    ]
    naano_vector += FUNCTIONAL_FP[aa_id]
    return naano_vector


def encoder_check(verbose=True):
    module = NAAnoEng(verbose=True)
    module.initialize()
    for aa_code, aa_vect in module.nAAno_emb.items():
        print(f"{aa_code} -- {aa_vect}")

    # check decoder and encoder
    for aa in AA_IDS:
        aa_str = aa
        aa_emb = get_embedding(aa)
        if aa_str == module.get_aa_id(aa_emb):
            if verbose:
                print(f"{aa_str}: str <-> vect aligned")
        else:
            raise ValueError(f"Ensure {aa} in nAAno_library is up to date")

encoder_check()  # note: all good

NAAnoEng initialized
A: str <-> vect aligned
C: str <-> vect aligned
D: str <-> vect aligned
E: str <-> vect aligned
F: str <-> vect aligned
G: str <-> vect aligned
H: str <-> vect aligned
I: str <-> vect aligned
K: str <-> vect aligned
L: str <-> vect aligned
M: str <-> vect aligned
N: str <-> vect aligned
P: str <-> vect aligned
Q: str <-> vect aligned
R: str <-> vect aligned
S: str <-> vect aligned
T: str <-> vect aligned
V: str <-> vect aligned
W: str <-> vect aligned
Y: str <-> vect aligned
VOID: str <-> vect aligned


In [31]:
import numpy as np

class RAAdialSeeker:

    def __init__(self, resolution, verbose=False):
        self.resolution = resolution
        self.verbose = verbose

        self.angstrom_lim = 33   # maximum angstrom range found + some extra for context enrichment
                                 # can always edit this later on
        self.angstrom_inc = float(33 / resolution)
        self.threshold = float(1 / resolution)  # standardize how we determine radial sequences
        #                           smallest distance is the base increment, not 0
        self.radius_levels = torch.arange(self.angstrom_inc, self.angstrom_lim + self.angstrom_inc, step=self.angstrom_inc)

        self.coord_layer = None # [1, 3] -> 3D vector

    def init_spAAtial(self):
        self._make_coord_layer()

    def _make_coord_layer(self):
        self.coord_layer = torch.randn(size=(1, 3))

    def radial_sequence(self, aa_seq, vect_seq):
        radial_seq = {}  # lookup table
        seen = []

        # sanity
        if len(aa_seq) != len(vect_seq):
            raise ValueError(f"string and vector sequences of {aa_seq} are different lengths")

        # iterate up resolution increments, if a molecule's coordinate is within like 1/resolution of an angstrom, append it's info
        # its radius is now the unique ID, so if we stumble on it again its not duplicated
        for level in self.radius_levels:
            radial_seq[level] = []
            for i in range(len(aa_seq)):
                num_id = (np.sqrt(sum(vect_seq[i] ** 2)), i)
                if self._dist_check(vect_seq[i], level) and (num_id not in seen):
                    radial_seq[level].append([aa_seq[i], vect_seq[i]])
                    seen.append(num_id)

        # create two VOID tokens (0,0,0) on both sides to denote stops and starts
        return [0, 0, 0] + radial_seq[::-1] + [0, 0, 0]  # we want to go from outside inward

    def _dist_check(self, vector, ang_radius):
        if abs(ang_radius - np.sqrt(sum(vector ** 2))) <= self.threshold:
            return True
        else:
            return False

def test_resolution():
    module = RAAdialSeeker(resolution = 100)
    module.init_spAAtial()
    for level in module.radius_levels:
        print(level)

test_resolution()

RAAdialSeeker layers aligned


True

In [6]:
mol_dim = 1024
map4_fprinter = MAP4(
    dimensions=mol_dim,
    radius=2,
    include_duplicated_shingles=True,
)

def eval_lipinski(smiles):
    """
    Use on user-input smiles, don't shut down inference run just flag molecule as non drug-like
    :param smiles:
    :return:
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES String")

    try:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Lipinski.NumHDonors(mol)
        h_acceptors = Lipinski.NumHAcceptors(mol)
    except Exception as e:
        raise ValueError(f"Error calculating Lipinski Descriptors: {e}")

    rules_passed = 0
    if mw <= 500: rules_passed += 1
    if logp <= 5: rules_passed += 1
    if h_donors <= 5: rules_passed += 1
    if h_acceptors <= 10: rules_passed += 1

    return rules_passed


def mol_from_smiles(smiles):
    """
    Extract molecular fingerprint from singular SMILES - fairly straightforward
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        map4_fp = map4_fprinter.calculate(mol=mol)

        return map4_fp

    except Exception as e:
        return None

In [54]:
# encode decode
def vect2idx(inc, vector):
    """Converts a torch.tensor of 3D vectors into their nearest shell-resolution cell index"""
    return torch.round(vector / inc).long()

def idx2vect(inc, idxs):
    """Converts a torch.tensor of indexes back into their original 3D vectors"""
    return idxs * inc

In [55]:
import torch

class Shell:
    """
    Basically a geometric tokenizer
    Performs coordinate discretization for any given 3D vector
    """
    def __init__(self, shell_resolution, max_angstroms):
        self.shell_resolution = shell_resolution + 1  # one more slot for the 0
        self.max_angstroms = max_angstroms  # make sure this is the same in RAAdialSeeker
        self.shell_increment = max_angstroms / shell_resolution

    def vect2idx(self, vector):
        """
        Converts a torch.tensor of 3D vectors into their nearest shell-resolution's index
        """
        idxs = torch.round(vector / self.shell_increment)
        idxs = torch.clamp(idxs, -self.shell_resolution, self.shell_resolution).long()
        return idxs

    def idx2vect(self, idxs):
        """Converts a torch.tensor of indexes back into their original 3D vectors"""
        return idxs * self.shell_increment


In [60]:
shell_test = Shell(shell_resolution=100, max_angstroms=33)
# okay so it appears 0 goes at the top..


tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.

In [26]:
# testing, generate radial sequence to get through
radial_module = RAAdialSeeker(resolution=1000, verbose=False)
aa_seq = test_sample["AA_IDs"]
vect_seq = test_sample["3d_struct"]
radial_seq = radial_module.radial_sequence(aa_seq=aa_seq, vect_seq=vect_seq)